# Notebook 6 – Interaction Features


## Learning objectives
By the end of this notebook, you will be able to:
- Explain interaction features and why relationships between variables can matter.
- Create product, sum, difference, ratio, polynomial, pairwise, and domain-based features.
- Detect invalid ratios such as division by zero.
- Decide when an interaction improves a model and when it adds unnecessary complexity.

## 1. What are interaction features?
An interaction feature combines two or more existing features to represent a relationship that may not be visible when the variables are considered separately.

Example: income alone and age alone may not describe a customer's financial capacity as well as `income × age` in a particular business context. The interaction does **not** automatically mean the feature is useful; it is a hypothesis that must be tested.

- Original features: `income`, `age`
- Interaction: `income_age_interaction = income × age`

Interaction features can help models represent relationships. They are often especially useful for linear models, which otherwise learn an additive relationship unless interactions are explicitly supplied. Tree-based models can learn many interactions through splits, so manually adding every possible interaction is often unnecessary.

## 2. Create a practical dataset

We will use a small customer and shopping dataset. The values are illustrative.

In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "customer_id": [101, 102, 103, 104, 105],
    "age": [22, 35, 45, 29, 55],
    "annual_income": [300000, 600000, 900000, 450000, 1200000],
    "experience_years": [1, 10, 20, 5, 30],
    "quantity": [2, 1, 5, 3, 4],
    "unit_price": [500, 1200, 250, 800, 600],
    "total_spending": [5000, 12000, 7500, 9600, 18000],
    "number_of_orders": [5, 8, 3, 6, 10]
})

df

**Expected observation:** Each row represents one customer. Numeric columns are the original input features. `customer_id` is an identifier and normally should not be used as a predictive feature.

## 3. Feature × Feature (multiplication)
Multiplication can represent combined intensity, scale, or exposure.

Examples:
- `income × age`: a constructed measure combining income and age.
- `quantity × unit_price`: total line-item value.
- `risk_score × exposure`: risk adjusted by exposure.

Use multiplication only when the product has a meaningful interpretation or helps the model.

In [ ]:
df["income_age"] = df["annual_income"] * df["age"]
df["line_item_value"] = df["quantity"] * df["unit_price"]

df[["age", "annual_income", "income_age",
    "quantity", "unit_price", "line_item_value"]]

**Expected output:** `line_item_value` is `[1000, 1200, 1250, 2400, 2400]`. This is a useful business calculation because quantity multiplied by unit price has a direct meaning.

## 4. Feature + Feature and Feature − Feature
Addition can combine components into a total. Subtraction can express a gap or difference.

Examples:
- `fixed_income + variable_income` → total income.
- `current_year_sales - previous_year_sales` → sales change.
- `age - experience_years` → approximate age when work experience began (only an approximation; education and career breaks matter).

In [ ]:
df["estimated_starting_age"] = df["age"] - df["experience_years"]
df["spending_per_order"] = df["total_spending"] / df["number_of_orders"]

df[["age", "experience_years", "estimated_starting_age",
    "total_spending", "number_of_orders", "spending_per_order"]]

## 5. Feature / Feature (ratios)
Ratios express relative quantities, efficiency, or average behaviour.

Examples:
- `experience_years / age`
- `total_spending / number_of_orders` → average spending per order
- `debt / income` → debt-to-income ratio

**Important:** Protect against zero denominators, missing values, and extreme ratios. A ratio is meaningful only when its numerator and denominator have compatible business meaning.

In [ ]:
df["experience_age_ratio"] = (
    df["experience_years"] / df["age"].replace(0, np.nan)
)
df["average_order_value"] = (
    df["total_spending"] /
    df["number_of_orders"].replace(0, np.nan)
)

df[["experience_age_ratio", "average_order_value"]]

## 6. Polynomial features
Polynomial features add powers of an existing variable, such as `age²`, and sometimes interactions such as `age × income`.

They can help model curved/non-linear patterns using a model that is otherwise linear. Higher degrees can create very large values, overfitting, and multicollinearity. Scaling may be important.

In [ ]:
df["age_squared"] = df["age"] ** 2
df[["age", "age_squared"]]

## 7. Pairwise interactions with scikit-learn
`PolynomialFeatures` can generate combinations systematically. `interaction_only=True` creates products between distinct features without powers such as `age²`.

Use a small, selected set of columns; do not feed identifiers or target columns into feature generation.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

base = df[["age", "annual_income"]]
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
expanded = pd.DataFrame(
    poly.fit_transform(base),
    columns=poly.get_feature_names_out(base.columns)
)
expanded

**Expected columns:** `age`, `annual_income`, and `age annual_income`. The third column is the pairwise product. The transformer is fitted on the supplied data here for demonstration; in a real ML workflow, fit preprocessing only on the training data, preferably inside a Pipeline.

## 8. Domain-based interactions
Domain knowledge helps decide which combinations are meaningful.

| Domain | Interaction | Possible meaning |
|---|---|---|
| Retail | quantity × unit price | line-item value |
| Banking | debt / income | debt burden |
| Insurance | claim amount × risk score | risk-weighted claim amount |
| HR | experience / age | relative career experience |
| Customer analytics | total spending / order count | average order value |

A domain-based feature should be checked with subject-matter experts and validated against the actual data definition.

## 9. When are interaction features useful?
Use them when:
1. There is a plausible business or scientific reason that one feature changes the effect of another.
2. A linear or additive model underfits a known relationship.
3. Validation results show a consistent improvement.
4. The feature is available at prediction time.

Avoid or limit them when:
- The interaction has no meaningful interpretation.
- You generate hundreds or thousands of combinations from a small dataset.
- The feature duplicates information or creates severe multicollinearity.
- It leaks target information or uses data unavailable at prediction time.
- Performance improves on training data but worsens on validation data.

## 10. Compare feature count and correlation

In [ ]:
original_features = ["age", "annual_income", "experience_years",
                     "quantity", "unit_price", "total_spending",
                     "number_of_orders"]
created_features = ["income_age", "line_item_value",
                    "estimated_starting_age", "spending_per_order",
                    "experience_age_ratio", "average_order_value",
                    "age_squared"]

print("Original feature count:", len(original_features))
print("Created feature count:", len(created_features))
print("Total numeric feature count:",
      len(original_features) + len(created_features))